# Filter surveys to the analysis population and build survey features

1. Keep survey rows whose `(Participant.Unique.ID, Year)` appears in the participant frame.
2. Audit each question block (`audit_block`) so no value is silently coerced.
3. Collapse the checkbox batteries into modeling features with `build_wide` → `data/to_use/survey_features_wide.csv`.

Feature definitions: [`docs/survey_features.md`](../../docs/survey_features.md).

In [ ]:
# Run from anywhere inside the repo: find the root, make it the working
# directory, and put it on sys.path so `pipeline/`, `descriptives/` and
# `paths.py` import cleanly. Data paths below are relative to the root.
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'paths.py').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [3]:
import pandas as pd
import numpy as np

In [ ]:
df1 = pd.read_csv('data/to_use/participant_info.csv', encoding='utf-8-sig')
df2 = pd.read_csv('data/processed/surveys.csv', encoding='utf-8-sig')

In [ ]:
df1['Participant.Unique.ID'] = df1['Participant.Unique.ID'].astype(str).str.strip()
df2['Participant.Unique.ID'] = df2['Participant.Unique.ID'].astype(str).str.strip()
df1['Year'] = df1['Year'].astype(int)
df2['Year'] = df2['Year'].astype(int)

In [ ]:
match_cols = ['Participant.Unique.ID', 'Year']
valid_combinations = df1.set_index(match_cols).index
filtered_df2 = df2[df2.set_index(match_cols).index.isin(valid_combinations)]

In [31]:
filtered_df2.shape

(17764, 636)

In [32]:
filtered_df2.to_csv('data/to_use/surveys.csv', index=False)

survey clean

In [4]:
import pandas as pd

In [5]:

df = pd.read_csv('data/to_use/surveys.csv')

In [6]:
df['SurveyType'] = df['SurveyType'].str.extract(r"^(PRE|POST)") # remove years -> now 2 unique values, PRE or POST

In [7]:
df[df['SurveyType'] == 'PRE'].isna().mean() * 100

Through which organization are you participating in the SYEP program?                                            12.463409
Through which organization are you participating in the SYEP program? - Other/I don't know: (Please fill in)     87.547850
How old are you?                                                                                                  0.000000
In which borough is your school located?                                                                          0.000000
How did you hear about this SYEP program? [Choose one answer]                                                     1.925242
                                                                                                                   ...    
Have your career goals changed because of your SYEP experience?                                                 100.000000
Which of the following do you plan to do in the next year? (Select all that apply)                              100.000000
How confident ar

In [8]:
df[df['SurveyType'] == 'POST'].isna().mean() * 100

Through which organization are you participating in the SYEP program?                                            10.752083
Through which organization are you participating in the SYEP program? - Other/I don't know: (Please fill in)    100.000000
How old are you?                                                                                                  0.000000
In which borough is your school located?                                                                          0.000000
How did you hear about this SYEP program? [Choose one answer]                                                   100.000000
                                                                                                                   ...    
Have your career goals changed because of your SYEP experience?                                                  79.790588
Which of the following do you plan to do in the next year? (Select all that apply)                               79.734294
How confident ar

In [59]:
from pipeline.survey_blocks import BlockSpec, build_block_frame, audit_block, discover_items

CF = BlockSpec(
    prefix="cf",
    stem="What would you have done this summer if you had not been in SYEP? [Check all that apply]",
    nota_label="None of the above",
    groups={
        "paid_alternative": ["A different part-time job",
                                "A different full-time job",
                                "Paid summer internship"],
        "unpaid_activity":  ["Volunteered", "Unpaid summer internship",
                                 "Summer camp", "Summer school"],
    },
    singles={"no_plans": "I had no other summer plans"},
    exclude_from_count=["I had no other summer plans"],
)

audit_block(df, CF)

[cf] block presence by Year:
Year
2022    0.5
2023    0.5
2024    0.5
2025    0.5 



,prefix,label,column,n_nonnull,n_selected,n_uncoercible,example_bad
0,cf,None of the above,What would you have done this summer if you ha...,8882,260,0,None
1,cf,A different part-time job,What would you have done this summer if you ha...,8882,4039,0,None
2,cf,A different full-time job,What would you have done this summer if you ha...,8882,1631,0,None
3,cf,Volunteered,What would you have done this summer if you ha...,6350,1313,0,None
4,cf,Paid summer internship,What would you have done this summer if you ha...,8882,1750,0,None
5,cf,Unpaid summer internship,What would you have done this summer if you ha...,8882,579,0,None
6,cf,Summer camp,What would you have done this summer if you ha...,8882,593,0,None
7,cf,Summer school,What would you have done this summer if you ha...,8882,663,0,None
8,cf,Spending time with friends,What would you have done this summer if you ha...,8882,3903,0,None
9,cf,Spending time with family,What would you have done this summer if you ha...,8882,4337,0,None


In [65]:
EXPECTATIONS = BlockSpec(
    prefix="expect",
    stem="Please indicate which of the following statements you agree with. [Check all that apply]",
    groups={
        "app_clarity": [
            "I understood all the steps I needed to complete to apply and enroll to the program",
            "I was able to get the information I needed to apply quickly and easily",
        ],
    },
    singles={
        "first_choice": "SYEP was my first choice for a summer program this year",
        "knew_what_to_expect": "When I started this program, I had a good idea of what it would be like",
    },
)

audit_block(df, EXPECTATIONS)

[expect] block presence by Year:
Year
2022    0.5
2023    0.5
2024    0.5
2025    0.5 



,prefix,label,column,n_nonnull,n_selected,n_uncoercible,example_bad
0,expect,None of the above,Please indicate which of the following stateme...,8882,183,0,None
1,expect,SYEP was my first choice for a summer program ...,Please indicate which of the following stateme...,8882,7079,0,None
2,expect,I understood all the steps I needed to complet...,Please indicate which of the following stateme...,8882,6749,0,None
3,expect,I was able to get the information I needed to ...,Please indicate which of the following stateme...,8882,5663,0,None
4,expect,"When I started this program, I had a good idea...",Please indicate which of the following stateme...,8882,4377,0,None
5,expect,I understand how to earn the full payment amount,Please indicate which of the following stateme...,8882,5380,0,None
6,expect,I understand the steps I need to take to recei...,Please indicate which of the following stateme...,8882,5747,0,None


In [73]:
BENEFITS = BlockSpec(
    prefix="benefit",
    stem="Besides earning money, what were the five biggest benefits you received from SYEP this past summer? [Select up to 5]",
    groups={
        "career_clarity": ["Identified careers I am interested in",
                           "Developed a career plan with clear next steps"],
        "job_readiness": ["Understood how to look for and get a job",
                          "Understood what employers are looking for",
                          "Developed a resume or other job application materials",
                          "Understood how to interact with professionals"],
        "self_efficacy": ["Developed new skills",
                          "Understood my own strengths",
                          "Felt motivated to seek a job"],
        "academic_credit": ["Earned course credit", "Earned academic credit"],
    },
    singles={"mentor_relationship": "Developed relationships with mentors/supervisors",
             "money_management": "Understood money management"},
    exclude_from_count=["Other (please specify)", "Decided to go to college"],
)

In [74]:
audit_block(df, BENEFITS)

[benefit] block presence by Year:
Year
2022    0.5
2023    0.5
2024    0.5
2025    0.5 



,prefix,label,column,n_nonnull,n_selected,n_uncoercible,example_bad
0,benefit,Developed new skills,"Besides earning money, what were the five bigg...",8882,4705,0,NaN
1,benefit,Understood how to look for and get a job,"Besides earning money, what were the five bigg...",8882,3420,0,NaN
2,benefit,Understood what employers are looking for,"Besides earning money, what were the five bigg...",8882,3497,0,NaN
3,benefit,Developed a resume or other job application ma...,"Besides earning money, what were the five bigg...",8882,2275,0,NaN
4,benefit,Identified careers I am interested in,"Besides earning money, what were the five bigg...",8882,2960,0,NaN
5,benefit,Developed a career plan with clear next steps,"Besides earning money, what were the five bigg...",8882,1621,0,NaN
6,benefit,Understood my own strengths,"Besides earning money, what were the five bigg...",8882,3789,0,NaN
7,benefit,Felt motivated to seek a job,"Besides earning money, what were the five bigg...",8882,2887,0,NaN
8,benefit,Earned course credit,"Besides earning money, what were the five bigg...",2532,42,0,NaN
9,benefit,Decided to go to college,"Besides earning money, what were the five bigg...",2532,128,0,NaN


In [12]:
from pipeline.build_survey_features import build_wide, report

wide = build_wide(df, include_secondary=False, how="inner")
report(wide)

Block coverage before split:
                  cf  expect  benefit
Year SurveyType                      
2022 POST        0.0     0.0      1.0
     PRE         1.0     1.0      0.0
2023 POST        0.0     0.0      1.0
     PRE         1.0     1.0      0.0
2024 POST        0.0     0.0      1.0
     PRE         1.0     1.0      0.0
2025 POST        0.0     0.0      1.0
     PRE         1.0     1.0      0.0 

dropped low-variance indicators: ['expect_nota', 'cf_nota', 'benefit_academic_credit']

rows: 8,882   unique people: 8,209
  ! 673 repeated people. Rows are NOT independent --
    cluster SEs on Participant.Unique.ID, or keep first year only.

rows per year:
Year
2022    2532
2023    1828
2024    2718
2025    1804

wave coverage:
wave_coverage
both          8882
left_only        0
right_only       0
  note: all rows matched -- input was already pruned to pairs,
        so no attrition comparison is possible from this frame.

feature coverage (want ~1.0 for in-scope features):
benefi

In [15]:
wide.to_csv("data/to_use/survey_features_wide.csv", index=False)